# 13 - Codility 2: sorting, stacks & queues, leader, maximum slice

**Source:** *Codility Lessons* chapters 6-9. Practice file: `codility/practice_03_stacks_leader_slices.py`
(+ the sorting tasks `distinct`, `triangle`, `max_product_of_three` in `practice_02_...`).

**Bugs / traps found in the PDF code** (all fixed below):
* `quadratic_max_slice`: `n = len(A), result = 0` creates a tuple - it is two separate statements.
* the lesson's maximum slice allows the **empty** slice (answer >= 0); the Codility task *MaxSliceSum* requires a
  **non-empty** slice - for all-negative input the answers differ.
* sorting snippets sort **in place**; copy first if the caller's data must be kept.

## 1. Sorting
| algorithm | time | memory | notes |
|---|---|---|---|
| selection sort | O(n^2) | O(1) | after k steps the first k are final (loop invariant) |
| counting sort | O(n + k) | O(k) | values in `0..k` |
| merge sort | O(n log n) | O(n) | divide, sort halves, merge |
| **Timsort** (`sorted`, `list.sort`) | O(n log n) worst, O(n) if already sorted | O(n) | **stable**, adaptive |

In real code always use `sorted(...)` / `.sort(key=..., reverse=...)`. Implement the others once to understand them.

In [1]:
import random
rng = random.Random(5)

def selection_sort(A):
    A = list(A)
    n = len(A)
    for k in range(n):
        minimal = k
        for j in range(k + 1, n):
            if A[j] < A[minimal]:
                minimal = j
        A[k], A[minimal] = A[minimal], A[k]
    return A

def counting_sort(A, k):
    count = [0] * (k + 1)
    for a in A:
        count[a] += 1
    out = []
    for value, c in enumerate(count):
        out.extend([value] * c)
    return out

def merge_sort(A):
    if len(A) <= 1:
        return list(A)
    mid = len(A) // 2
    left, right = merge_sort(A[:mid]), merge_sort(A[mid:])
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:                # <= keeps the sort stable
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    return out + left[i:] + right[j:]

for _ in range(200):
    A = [rng.randint(0, 20) for _ in range(rng.randint(0, 30))]
    expected = sorted(A)
    assert selection_sort(A) == counting_sort(A, 20) == merge_sort(A) == expected
print("all sorts agree with sorted()")

all sorts agree with sorted()


In [2]:
# Sorting idioms
words = ["banana", "Apple", "cherry", "apple"]
print(sorted(words), sorted(words, key=str.lower), sorted(words, key=len, reverse=True))
people = [("Ann", 30), ("Bob", 25), ("Cy", 30)]
print(sorted(people, key=lambda p: (-p[1], p[0])))        # by age desc, then name (stable multi-key)

['Apple', 'apple', 'banana', 'cherry'] ['Apple', 'apple', 'banana', 'cherry'] ['banana', 'cherry', 'Apple', 'apple']
[('Ann', 30), ('Cy', 30), ('Bob', 25)]


### Sorting as a tool: *distinct*, *triangle*, *max product of three*
Many problems become trivial once the data is sorted because equal / related items become **neighbours**.

* distinct values: count positions where `A[k] != A[k-1]` (or `len(set(A))` - O(n) average).
* triangle: if any triple works, a triple of *consecutive* sorted values works.
* max product of three: the answer is one of two candidates - the three largest, or the two smallest (negatives!) times the largest.

In [3]:
def distinct_sorted(A):
    A = sorted(A)
    return 0 if not A else 1 + sum(A[k] != A[k - 1] for k in range(1, len(A)))

assert distinct_sorted([2, 1, 1, 2, 3, 1]) == len({2, 1, 1, 2, 3, 1}) == 3 and distinct_sorted([]) == 0

## 2. Stacks and queues
* **Stack** (LIFO): `push` / `pop` at the top. Python: a `list` with `append()` / `pop()` (O(1)).
* **Queue** (FIFO): `push` at the back / `pop` from the front. Python: `collections.deque` with `append()` /
  `popleft()` (O(1)); `list.pop(0)` is O(n)!
* The PDF's fixed-size array + `global size` version is how it is done in C; in Python use the built-in structures,
  or a small class.

In [4]:
from collections import deque

stack = []
stack.append(8); stack.append(4); stack.append(7)
assert stack.pop() == 7 and stack[-1] == 4                # peek = stack[-1]

queue = deque()
queue.append(8); queue.append(4); queue.append(6)
assert queue.popleft() == 8 and queue[0] == 4

class CircularQueue:
    """Fixed capacity queue on a cyclic buffer - the PDF's version, made safe."""
    def __init__(self, capacity):
        self._buf = [None] * (capacity + 1)               # one slot stays empty to tell full from empty
        self._head = self._tail = 0
    def __len__(self):
        return (self._tail - self._head) % len(self._buf)
    def empty(self):
        return self._head == self._tail
    def push(self, x):
        if len(self) == len(self._buf) - 1:
            raise OverflowError("queue is full")
        self._buf[self._tail] = x
        self._tail = (self._tail + 1) % len(self._buf)
    def pop(self):
        if self.empty():
            raise IndexError("pop from empty queue")
        x = self._buf[self._head]
        self._head = (self._head + 1) % len(self._buf)
        return x

q = CircularQueue(3)
for v in (1, 2, 3):
    q.push(v)
assert [q.pop(), q.pop()] == [1, 2]
q.push(4); q.push(5)                                       # wraps around the buffer
assert [q.pop(), q.pop(), q.pop()] == [3, 4, 5] and q.empty()

### Grocery store (PDF 7.3): minimal initial queue
Simulate the size; the most negative value it reaches is the answer. O(n) time, O(1) space.

In [5]:
def grocery_store(A):
    size = result = 0
    for a in A:
        size += 1 if a == 0 else -1
        result = max(result, -size)
    return result

assert grocery_store([1, 1, 0, 1, 0, 1]) == 2      # 1,1 need two people; 0 -> +1; 1 -> 0 ... etc.

### Typical stack problems (-> `practice_03`)
* **brackets** - push openers, on a closer the top must match.
* **stone wall** - monotonic stack of heights; each *new* height pushed = one new block.
* **fish** - stack of downstream fish; an upstream fish fights the top until it wins or loses.
* **monotonic stack** in general: "next greater element", "largest rectangle in a histogram".

In [6]:
def brackets_demo(S):
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in S:
        if ch in "([{":
            stack.append(ch)
        elif not stack or stack.pop() != pairs[ch]:
            return 0
    return int(not stack)

assert [brackets_demo(s) for s in ("{[()()]}", "([)()]", "", "((")] == [1, 0, 1, 0]

## 3. Leader
The **leader** occurs in **more than n/2** positions; there is at most one.

1. O(n^2): count every element.
2. O(n log n): sort -> the leader must be the median `A[n//2]`.
3. **O(n) - Boyer-Moore voting:** remove pairs of different values; the leader (if it exists) survives. Keep only a
   candidate and a counter. A second pass verifies the candidate.
4. O(n) with `Counter` (dict) - easiest in Python.

In [7]:
from collections import Counter

def slow_leader(A):
    n = len(A)
    leader = -1
    for k in range(n):
        candidate = A[k]
        count = sum(1 for x in A if x == candidate)
        if count > n // 2:
            leader = candidate
    return leader

def fast_leader(A):
    A = sorted(A)                                  # copy! the PDF sorts the caller's list
    if not A:
        return -1
    candidate = A[len(A) // 2]
    return candidate if A.count(candidate) > len(A) // 2 else -1

def golden_leader(A):
    size, value = 0, None
    for x in A:
        if size == 0:
            size, value = 1, x
        elif value == x:
            size += 1
        else:
            size -= 1
    candidate = value if size > 0 else None
    if candidate is None:
        return -1
    return candidate if sum(x == candidate for x in A) > len(A) // 2 else -1

def counter_leader(A):
    if not A:
        return -1
    v, c = Counter(A).most_common(1)[0]
    return v if c > len(A) // 2 else -1

assert golden_leader([6, 8, 4, 6, 8, 6, 6]) == 6
for _ in range(1000):
    A = [rng.randint(0, 3) for _ in range(rng.randint(0, 9))]
    assert slow_leader(A) == fast_leader(A) == golden_leader(A) == counter_leader(A), A
print("four leader algorithms agree")

four leader algorithms agree


Pitfalls: *exactly* half is **not** a leader (`>` not `>=`); with `size > 0` the candidate may still fail the verification
pass (`[1, 2, 3]` -> candidate 3, count 1); the PDF's `slow_leader` returns the last candidate satisfying the test,
which is fine because there is at most one.

**Equi-leader:** a split is good if both sides have the *same* leader. That leader must be the leader of the whole array
-> find it once, then sweep and keep a left count (prefix sums!).

## 4. Maximum slice
Find the contiguous slice with the largest sum.

* O(n^3): all `(p, q)` and sum each.
* O(n^2): prefix sums or a running sum.
* **O(n) - Kadane:** `best_ending_here = max(0, best_ending_here + a)`; `best = max(best, best_ending_here)`.

The idea ("optimal solution for the prefix, extended by one element") is **dynamic programming**.

In [8]:
def slow_max_slice(A):                      # O(n^3)
    result = 0
    for p in range(len(A)):
        for q in range(p, len(A)):
            result = max(result, sum(A[p:q + 1]))
    return result

def quadratic_max_slice(A):                 # O(n^2)  (the PDF's version has the `n = len(A), result = 0` bug)
    n, result = len(A), 0
    for p in range(n):
        total = 0
        for q in range(p, n):
            total += A[q]
            result = max(result, total)
    return result

def golden_max_slice(A):                    # O(n); empty slice allowed -> result >= 0
    max_ending = max_slice = 0
    for a in A:
        max_ending = max(0, max_ending + a)
        max_slice = max(max_slice, max_ending)
    return max_slice

def max_slice_nonempty(A):                  # O(n); Codility MaxSliceSum: at least one element
    best = cur = A[0]
    for a in A[1:]:
        cur = max(a, cur + a)
        best = max(best, cur)
    return best

assert golden_max_slice([5, -7, 3, 5, -2, 4, -1]) == 10
assert golden_max_slice([-3, -1]) == 0 and max_slice_nonempty([-3, -1]) == -1     # the difference
for _ in range(500):
    A = [rng.randint(-9, 9) for _ in range(rng.randint(1, 10))]
    assert slow_max_slice(A) == quadratic_max_slice(A) == golden_max_slice(A)
print("max-slice variants agree")

max-slice variants agree


In [9]:
# Which slice? Track the indices as well
def best_slice_with_indices(A):
    best, best_range = A[0], (0, 0)
    cur, start = A[0], 0
    for i in range(1, len(A)):
        if cur < 0:
            cur, start = A[i], i
        else:
            cur += A[i]
        if cur > best:
            best, best_range = cur, (start, i)
    return best, best_range

print(best_slice_with_indices([5, -7, 3, 5, -2, 4, -1]))

(10, (2, 5))


**Variants** (-> `practice_03`): *max profit* (maximum slice of the day-to-day differences, or prefix minimum),
*max double slice sum* (best slice ending before `Y` + best slice starting after `Y`, computed with two sweeps),
*circular* max slice (total - min slice), *maximum product subarray* (track max and min).

## Self-check
1. Why is `list.pop(0)` in a loop dangerous? What replaces it?
2. Show that Boyer-Moore cannot give a false *positive* after the verification pass but can give a false *candidate*.
3. What changes in Kadane if the slice must be non-empty? If it must have length >= 2?
4. Which sorting algorithm would you pick for 10^6 integers in `0..1000`? For 10^6 arbitrary objects?